<a href="https://colab.research.google.com/github/AnjaliMenezes/open-bapm/blob/main/Adjusted_APM_for_WNBA_Version_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install packages



In [ ]:
install.packages(c("wehoop", "dplyr", "tidyr", "Matrix", "glmnet", "glue", "tibble"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘snakecase’, ‘iterators’, ‘janitor’, ‘progressr’, ‘RcppParallel’, ‘stringdist’, ‘foreach’, ‘shape’, ‘RcppEigen’




In [ ]:
#Load Data & Aggregate Stints
wnba_pbp <- wehoop::load_wnba_pbp()
wnba_player_box <- wehoop::load_wnba_player_box()

# Starters identification
starters <- wnba_player_box |>
  dplyr::filter(starter) |>
  dplyr::mutate(
    score_diff_final = (1 - 2 * (home_away == "away")) * (team_score - opponent_team_score)
  ) |>
  dplyr::select(game_id, home_away, athlete_id, score_diff_final)

# Substitutions extraction
substitutions <- wnba_pbp |>
  dplyr::filter(type_text == "Substitution") |>
  dplyr::mutate(
    time_start = 10 * (qtr - 1) + (9 - clock_minutes) + (60 - clock_seconds) / 60,
    time_remaining = end_game_seconds_remaining / 60,
    score_diff_start = home_score - away_score
  ) |>
  dplyr::select(game_id, time_start, time_remaining, score_diff_start, athlete_id_1, athlete_id_2)

player_on <- starters |>
  dplyr::group_by(game_id, home_away) |>
  dplyr::mutate(index = 1:dplyr::n(), time_start = 0, score_diff_start = 0, time_remaining = NA) |>
  dplyr::ungroup()

for (i in 1:nrow(substitutions)) {
  new_player_on <- player_on |>
    dplyr::filter(
      game_id == substitutions$game_id[i],
      athlete_id == substitutions$athlete_id_2[i]
    ) |>
    dplyr::slice(dplyr::n()) |>
    dplyr::mutate(
      athlete_id = substitutions$athlete_id_1[i],
      time_start = substitutions$time_start[i],
      time_remaining = substitutions$time_remaining[i],
      score_diff_start = substitutions$score_diff_start[i]
    )
  player_on <- dplyr::bind_rows(player_on, new_player_on)
}

lineups <- player_on |>
  dplyr::group_by(game_id, time_start, time_remaining, score_diff_start, score_diff_final, home_away, index) |>
  dplyr::summarize(athlete_id = athlete_id[dplyr::n()], .groups = "drop") |>
  dplyr::mutate(name = glue::glue("{home_away}_{index}")) |>
  dplyr::select(-home_away, -index) |>
  tidyr::pivot_wider(names_from = name, values_from = athlete_id) |>
  tidyr::fill(dplyr::matches("^away|^home"), .direction = "down")

athlete <- wnba_player_box |>
  dplyr::count(athlete_id, athlete_display_name, team_abbreviation) |>
  dplyr::group_by(athlete_id) |>
  dplyr::arrange(-n) |>
  dplyr::slice(1) |>
  dplyr::select(-n) |>
  dplyr::ungroup()

In [ ]:
head(lineups)                                                                                     #
head(athlete)

game_id,time_start,time_remaining,score_diff_start,score_diff_final,away_1,away_2,away_3,away_4,away_5,home_1,home_2,home_3,home_4,home_5
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
401856890,0.000000,NA,0,31,4433635,4684384,2490553,3922628,4433514,2998928,2999101,2593770,4038379,5345320
401856890,4.466667,35.38333,10,31,4703609,4684384,2490553,3922628,4433514,2998928,2999101,2593770,4038379,5345320
401856890,4.466667,35.53333,10,31,4703609,4684384,2490553,3922628,4433514,2998928,2327695,4337216,4038379,5345320
401856890,4.800000,35.13333,10,31,4703609,4684384,2490553,3922628,4433514,2998928,2327695,4337216,2566110,5345320
401856890,5.800000,33.91667,15,31,4703609,4398966,2490553,3922628,4433514,2998928,2327695,4337216,2566110,5345320
401856890,6.266667,33.73333,15,31,4703609,4398966,2490553,3922628,4433514,2998928,2327695,4337216,2566110,4038379


athlete_id,athlete_display_name,team_abbreviation
<int>,<chr>,<chr>
869,DeWanna Bonner,PHX
887,Sami Whitcomb,PHX
924,Alysha Clark,DAL
981,Courtney Vandersloot,CHI
1054,Tiffany Hayes,GS
1068,Nneka Ogwumike,LA


In [ ]:
# Stint calculations
data <- lineups |>
  dplyr::group_by(game_id) |>
  dplyr::mutate(
    minutes = dplyr::coalesce(dplyr::lead(time_start, 1) - time_start, time_remaining),
    score_diff = dplyr::coalesce(dplyr::lead(score_diff_start, 1), score_diff_final) - score_diff_start
  ) |>
  dplyr::ungroup() |>
  dplyr::filter(minutes > 0)

data_long <- data |>
  dplyr::mutate(stint = 1:dplyr::n()) |>
  dplyr::select(game_id, stint, minutes, score_diff, dplyr::matches("^away|^home")) |>
  tidyr::pivot_longer(cols = c(dplyr::matches("^away|^home")), values_to = "athlete_id")

athlete_summary <- data_long |>
  dplyr::group_by(athlete_id) |>
  dplyr::summarize(
    minutes = sum(minutes),
    plus_minus = sum((1 - 2 * grepl("away", name)) * score_diff),
    .groups = "drop"
  )

In [ ]:
unique_athletes <- sort(unique(data_long$athlete_id))
n_players <- length(unique_athletes)

X_data <- data_long |>
  dplyr::mutate(
    row = stint,
    player_idx = match(athlete_id, unique_athletes),
    # Separate Offense vs Defense columns:
    # Home team = Offense (+1 in Offense col), Away team = Defense (-1 in Defense col)
    col_offense = ifelse(substring(name, 1, 4) == "home", player_idx, player_idx + n_players),
    col_defense = ifelse(substring(name, 1, 4) == "away", player_idx, player_idx + n_players),
    is_home = substring(name, 1, 4) == "home"
  )

# Overall RAPM Design Matrix (Single rating per player + Home Court Advantage)
X_data_overall <- data_long |>
  dplyr::mutate(
    row = stint,
    column = match(athlete_id, unique_athletes),
    value = ifelse(substring(name, 1, 4) == "home", 1, -1)
  )

x_mat <- Matrix::sparseMatrix(
  i = X_data_overall$row,
  j = X_data_overall$column,
  x = X_data_overall$value
)

y <- data$score_diff / data$minutes
w <- data$minutes

# -----------------------------------------------------------------------------
# 3. Fit Cross-Validated Ridge Regression (Adjusted APM)
# -----------------------------------------------------------------------------
set.seed(42)
model_adjusted_rapm <- glmnet::cv.glmnet(
  x = x_mat,
  y = y,
  weights = w,
  alpha = 0,              # L2 Ridge Penalty
  intercept = TRUE,       # Estimates Home-Court Advantage (beta_0)
  standardize = FALSE,    # Retain binary indicator scale (+1/-1)
  nfolds = 10
)

# Extract Home-Court Advantage per minute (beta_0)
hca_per_minute <- coef(model_adjusted_rapm, s = "lambda.min")[1, 1]
hca_per_40_min <- hca_per_minute * 40

In [ ]:
athlete_coef_rapm <- tibble::tibble(
  athlete_id = unique_athletes,
  rapm_per_min = coef(model_adjusted_rapm, s = "lambda.min")[-1, 1]
) |>
  dplyr::mutate(
    # Scale from points per minute to points per 40 minutes (standard WNBA game length)
    rapm_40 = rapm_per_min * 40
  )

# Merge with Summary & Athlete metadata
final_rapm_leaderboard <- athlete_summary |>
  dplyr::inner_join(athlete_coef_rapm, by = "athlete_id") |>
  dplyr::left_join(athlete, by = "athlete_id") |>
  dplyr::filter(minutes >= 100) |> # Filter out sample-size noise (<100 mins)
  dplyr::arrange(-rapm_40)

# Display Top 10 Players
head(final_rapm_leaderboard, 10)

athlete_id,minutes,plus_minus,rapm_per_min,rapm_40,athlete_display_name,team_abbreviation
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
3149391,1202.4267,295,0.09254389,3.701756,A'ja Wilson,LV
4065870,1316.4283,278,0.08159229,3.263692,Jackie Young,LV
4398935,1136.0083,275,0.07841639,3.136656,Veronica Burton,GS
2529130,1195.7133,297,0.06559993,2.623997,Natasha Howard,MIN
3058901,1343.1250,258,0.06192939,2.477176,Allisha Gray,ATL
4433791,1197.2333,277,0.06186507,2.474603,Olivia Miles,MIN
4065780,785.3067,218,0.05806592,2.322637,Kaila Charles,GS
4432831,1012.3617,210,0.05781530,2.312612,Aliyah Boston,IND
3142191,1355.4733,245,0.05747287,2.298915,Kelsey Mitchell,IND
